In [0]:
%pip install faker

In [0]:
dbutils.library.restartPython()

In [0]:
# COMMAND ----------
# Generate the source data (reuses your existing generator, no Snowflake dependency)
import sys
sys.path.append("/Workspace/Users/youngli.hong2@gmail.com/restaurant_analytics")
from generate_restaurant_data import main
main()

In [0]:
# COMMAND ----------
# Load order_items.csv into a Spark DataFrame
order_items = spark.read.csv(
    "file:/Workspace/Users/youngli.hong2@gmail.com/restaurant_analytics/raw_data/order_items.csv",
    header=True,
    inferSchema=True,
)
order_items.show(5)


In [0]:
# COMMAND ----------
from pyspark.sql.functions import col

# This is the direct PySpark port of int_order_items_priced.sql:
# revenue = quantity * unit_price, using the SNAPSHOTTED unit_price already
# on each row — deliberately not joined back to menu_items.base_price,
# since that would give current price, not transaction-time price.
priced = order_items.withColumn(
    "revenue", col("quantity") * col("unit_price")
)

priced.select("order_item_id", "order_id", "menu_item_id", "quantity", "unit_price", "revenue").show(10)


In [0]:
# COMMAND ----------
# Sanity check: total revenue should match what dbt/Snowflake computed
total_revenue = priced.agg({"revenue": "sum"}).collect()[0][0]
print(f"Total revenue (PySpark): {total_revenue:,.2f}")